# ML-07 — Baseline Action Score and Top-20 Review

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

In [5]:
rule_text = (
    "A content item is worth a human action if it is visible (non-trivial recent impressions), "
    "appears stale (it has not been updated recently), and its search performance is slipping "
    "(negative trend in traffic or position)."
)
reason_codes = {
    "visible_only": "item has impressions but is not stale or slipping",
    "stale_only": "item is stale but has low visibility",
    "slipping_only": "item is slipping but has low visibility / is not stale",
    "stale_visible": "visible and stale (good candidate)",
    "visible_slipping": "visible and slipping (good candidate)",
    "stale_visible_slipping": "all three signals (highest priority)",
}

print(rule_text)
print("\nReason codes this rule can output:")
for code, description in reason_codes.items():
    print(f"- {code}: {description}")

A content item is worth a human action if it is visible (non-trivial recent impressions), appears stale (it has not been updated recently), and its search performance is slipping (negative trend in traffic or position).

Reason codes this rule can output:
- visible_only: item has impressions but is not stale or slipping
- stale_only: item is stale but has low visibility
- slipping_only: item is slipping but has low visibility / is not stale
- stale_visible: visible and stale (good candidate)
- visible_slipping: visible and slipping (good candidate)
- stale_visible_slipping: all three signals (highest priority)


## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

In [7]:
# Section 2: Build the ranked queue and write work/outputs/baseline_action_score.csv
import os
from pathlib import Path
import numpy as np
import pandas as pd

# Try common cached / sample paths first.
root = Path.cwd().resolve()
candidate_paths = [
    Path("work/outputs/features.parquet"),
    Path("work/outputs/content_snapshot.parquet"),
    Path("work/inputs/features.parquet"),
    Path("work/inputs/content_sample.parquet"),
    Path("data/content_sample.parquet"),
    Path("data/raw/content_refresh_anonymized.csv"),
    Path("../../data/raw/content_refresh_anonymized.csv"),
    Path("../data/raw/content_refresh_anonymized.csv"),
    root / "data/raw/content_refresh_anonymized.csv",
    root.parent / "data/raw/content_refresh_anonymized.csv",
    root.parent.parent / "data/raw/content_refresh_anonymized.csv",
]

df = None
for path in candidate_paths:
    if path.exists():
        try:
            if path.suffix == ".parquet":
                df = pd.read_parquet(path)
            else:
                df = pd.read_csv(path)
            print(f"Loaded data from {path} — {len(df):,} rows")
            break
        except Exception as e:
            print(f"Found {path} but failed to read: {e}")

if df is None:
    raise FileNotFoundError(
        "No local feature file found in expected locations. "
        "Place your feature table at one of the candidate paths or update the path in this cell."
    )

print("Columns (sample):", list(df.columns)[:50])

# helpers to find candidate columns
def find_col(*names):
    for n in names:
        if n in df.columns:
            return n
    return None

impr_col = find_col("impressions_last_30d", "impressions_prev30", "impressions_30d", "impressions")
pos_col = find_col("avg_position", "gsc_avg_position", "position")
trend_col = find_col("trend_pct", "trend_percent", "trend")
updated_col = find_col("days_since_last_update", "days_since_update", "days_old", "age_days", "last_updated_days")
label_col = find_col("is_declining_label", "label", "is_declining")

# Fill sensible defaults / safe operations
df = df.copy()
df["impressions_prev30_safe"] = df.get(impr_col, pd.Series(0, index=df.index)).fillna(0).astype(float)
df["gsc_avg_position_safe"] = df.get(pos_col, pd.Series(np.nan, index=df.index)).astype(float)
df["trend_pct_safe"] = df.get(trend_col, pd.Series(np.nan, index=df.index)).astype(float)
df["days_since_update_safe"] = df.get(updated_col, pd.Series(np.nan, index=df.index)).astype(float)

# Flags (no fitted weights — simple thresholds)
visible = (df["impressions_prev30_safe"] >= 500).astype(int)
stale = (df["days_since_update_safe"] >= 180).fillna(0).astype(int)
slipping = (df["trend_pct_safe"] <= -5).fillna(0).astype(int)

df["_visible_flag"] = visible
df["_stale_flag"] = stale
df["_slipping_flag"] = slipping

df["baseline_score"] = (
    df["_visible_flag"]
    * (1 + df["_stale_flag"])
    * (1 + df["_slipping_flag"])
    * (df["impressions_prev30_safe"] + 1)
)

# Reason code assignment (readable strings)
def reason_code(row):
    v, s, l = row["_visible_flag"], row["_stale_flag"], row["_slipping_flag"]
    if v and s and l:
        return "stale_visible_slipping"
    if v and s:
        return "stale_visible"
    if v and l:
        return "visible_slipping"
    if v:
        return "visible_only"
    if s and l:
        return "stale_slipping"
    if s:
        return "stale_only"
    if l:
        return "slipping_only"
    return "none"

df["reason_code"] = df.apply(reason_code, axis=1)

# Confidence note column based on signal strength
def confidence_note(row):
    impressions = row["impressions_prev30_safe"]
    trend = row["trend_pct_safe"]
    if row["reason_code"] == "stale_visible_slipping":
        if impressions >= 2000 and pd.notna(trend) and trend <= -20:
            return "high_confidence: strong traffic and large decline"
        return "medium_confidence: multiple signals"
    if row["reason_code"] in ("stale_visible", "visible_slipping"):
        return "medium_confidence"
    if row["reason_code"] in ("visible_only", "stale_only", "slipping_only"):
        return "low_confidence"
    return "no_signal"

df["confidence_note"] = df.apply(confidence_note, axis=1)

# Rank everything and save essential columns
df = df.sort_values("baseline_score", ascending=False)
out_cols = ["content_id", "client_id"] if "content_id" in df.columns else df.columns[:2].tolist()
out_cols = list(dict.fromkeys(out_cols + ["baseline_score", "reason_code", "confidence_note"]))
if label_col:
    out_cols.append(label_col)

os.makedirs("work/outputs", exist_ok=True)
csv_path = "work/outputs/baseline_action_score.csv"
df[out_cols].to_csv(csv_path, index=False)
print(f"Wrote ranked queue to {csv_path} — top rows:")
print(df[out_cols].head(10).to_string(index=False))

if label_col and label_col in df.columns:
    def precision_at_k(scores, labels, k):
        order = np.argsort(-np.asarray(scores))
        return np.asarray(labels)[order[:k]].mean()
    labels = df[label_col].fillna(0).astype(int).values
    scores = df["baseline_score"].values
    for k in [10, 20, 50]:
        if len(df) >= k:
            print(f"precision@{k}: {precision_at_k(scores, labels, k):.3f}")
    print("base rate (labels.mean):", labels.mean())
else:
    print("No label column detected — precision@K not computed.")

Loaded data from ..\..\data\raw\content_refresh_anonymized.csv — 30,000 rows
Columns (sample): ['content_id', 'client_id', 'search_volume', 'competition', 'competition_level', 'cpc', 'content_type', 'main_intent', 'word_count', 'char_count', 'provider_used', 'model_used', 'impressions_90d', 'clicks_90d', 'pageviews_90d', 'sessions_90d', 'users_90d', 'engaged_sessions_90d', 'ai_sessions_90d', 'scroll_events_90d', 'days_with_impressions', 'days_with_sessions', 'impressions_last_30d', 'clicks_last_30d', 'sessions_last_30d', 'impressions_prev_30d', 'clicks_prev_30d', 'sessions_prev_30d', 'content_age_days', 'age_tier', 'age_tier_order', 'days_since_last_update', 'freshness_tier', 'word_count_tier', 'char_count_tier', 'ctr', 'avg_position', 'engagement_rate', 'scroll_rate', 'ai_traffic_pct', 'impression_tier', 'position_tier', 'trend_direction', 'trend_pct']
Wrote ranked queue to work/outputs/baseline_action_score.csv — top rows:
          content_id         client_id  baseline_score      r

## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

In [8]:
# Section 3: Top-20 review
top20 = df.head(20).copy()
display_cols = ["content_id", "client_id", "baseline_score", "reason_code", "confidence_note",
                "impressions_prev30_safe", "gsc_avg_position_safe", "trend_pct_safe", "days_since_update_safe"]
print("Top-20 (compact):")
print(top20[display_cols].to_string(index=False))

# For each top 20 produce the short review: action, reason code, confidence note, and what would make it wrong.
reviews = []
for _, row in top20.iterrows():
    action = "Review & update content" if row["reason_code"] in ("stale_visible", "stale_visible_slipping", "visible_slipping") else "Monitor"
    reason = row["reason_code"]
    confidence = row["confidence_note"]
    # what would make it wrong (brief checklist)
    wrong_reasons = []
    if row["impressions_prev30_safe"] < 600:
        wrong_reasons.append("low/latest impressions (no real exposure)")
    if pd.isna(row["trend_pct_safe"]):
        wrong_reasons.append("no reliable trend metric")
    if pd.isna(row["days_since_update_safe"]):
        wrong_reasons.append("no update date available")
    if int(row["_visible_flag"]) == 0:
        wrong_reasons.append("not actually visible / impressions tool mismatch")
    reviews.append({
        "content_id": row.get("content_id", None),
        "client_id": row.get("client_id", None),
        "action": action,
        "reason_code": reason,
        "confidence": confidence,
        "what_makes_it_wrong": "; ".join(wrong_reasons) or "none_obvious"
    })

reviews_df = pd.DataFrame(reviews)
print("\nTop-20 brief reviews:")
print(reviews_df.to_string(index=False))
# Save a small human-review file
reviews_df.to_csv("work/outputs/baseline_top20_review.csv", index=False)
print("Saved top-20 review to work/outputs/baseline_top20_review.csv")

Top-20 (compact):
          content_id         client_id  baseline_score      reason_code   confidence_note  impressions_prev30_safe  gsc_avg_position_safe  trend_pct_safe  days_since_update_safe
content_5fe46e04994d client_4e07408562        241584.0 visible_slipping medium_confidence                 120791.0                    4.2           -44.8                   104.0
content_db5989a78dd3 client_4e07408562        238797.0     visible_only    low_confidence                 238796.0                    5.4           556.2                    20.0
content_9532f197bbc8 client_4e07408562        218636.0 visible_slipping medium_confidence                 109317.0                    2.0           -37.3                   104.0
content_1a9e894be2e2 client_19581e27de        215974.0 visible_slipping medium_confidence                 107986.0                    4.0           -27.0                    22.0
content_a023517539fe client_6208ef0f77        212016.0     visible_only    low_confidence   

## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

In [9]:
# Section 4: Weak picks + leakage check

# 1) Flag weak picks in the top 100: items where the rule relied on a single weak signal
top100 = df.head(100).copy()
weak_mask = (
    ((top100["_visible_flag"] == 1) & (top100["_stale_flag"] == 0) & (top100["_slipping_flag"] == 0)) |
    ((top100["_stale_flag"] == 1) & (top100["_visible_flag"] == 0) & (top100["_slipping_flag"] == 0)) |
    ((top100["_slipping_flag"] == 1) & (top100["_visible_flag"] == 0) & (top100["_stale_flag"] == 0))
)
weak_picks = top100[weak_mask]
print(f"{len(weak_picks)} weak picks in top 100 (rely on a single weak signal). Example rows:")
print(weak_picks[["content_id", "client_id", "baseline_score", "reason_code",
                   "impressions_prev30_safe", "gsc_avg_position_safe", "trend_pct_safe",
                   "days_since_update_safe"]].head(10).to_string(index=False))

# 2) Leakage check: search for suspicious column names and check date alignment
suspect_cols = [c for c in df.columns if any(k in c.lower() for k in ["future", "leak", "label", "next_", "_next", "is_declining_label", "window"])]
print("\nSuspect column names (manual check):", suspect_cols)

# Check for columns that look like direct labels or future windows
label_like = [c for c in df.columns if "label" in c.lower() or c.lower().startswith("is_declining")]
if label_like:
    print("Warning: label-like columns present — ensure they are NOT used as features. Columns:", label_like)

# Simple date-window leak check: if you have per-row report_date or label_date, ensure we don't use future metrics
date_cols = [c for c in df.columns if "date" in c.lower() or "report_date" in c.lower()]
print("Date-like columns observed:", date_cols)

# quick manual test: ensure we didn't accidentally use trend_direction or trend_pct as features that are derived from label
derived_cols = [c for c in df.columns if "trend_direction" in c.lower() or "trend_pct" in c.lower()]
if derived_cols:
    print("Note: trend_pct/direction detected in table; flyrank-data SKILL warns these are tied to labels — treat them carefully and avoid using derived features that are computed from the same window as the label.")

# Save a short leakage check report
leak_report = {
    "weak_picks_in_top100": int(len(weak_picks)),
    "label_like_columns_found": label_like,
    "suspect_columns": suspect_cols,
    "date_columns": date_cols,
}
import json
with open("work/outputs/baseline_leakage_check.json", "w") as f:
    json.dump(leak_report, f, indent=2)
print("Saved leakage check summary to work/outputs/baseline_leakage_check.json")

17 weak picks in top 100 (rely on a single weak signal). Example rows:
          content_id         client_id  baseline_score  reason_code  impressions_prev30_safe  gsc_avg_position_safe  trend_pct_safe  days_since_update_safe
content_db5989a78dd3 client_4e07408562        238797.0 visible_only                 238796.0                    5.4           556.2                    20.0
content_a023517539fe client_6208ef0f77        212016.0 visible_only                 212015.0                   85.8         27907.3                    20.0
content_2cb567c3c89b client_6208ef0f77        197678.0 visible_only                 197677.0                   22.2            23.1                    48.0
content_aaef01a50def client_19581e27de        170560.0 visible_only                 170559.0                    5.4            -4.0                    22.0
content_8451fc6f034d client_d029fa3a95        168959.0 visible_only                 168958.0                    2.3            73.1                  

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.